# Módulo 3 · Agente híbrido: RAG sobre el catálogo de datos + SQL en BigQuery + informe y política
**Caso:** un **analista financiero** (agente) al que le preguntas por los datos de la empresa. Elige **qué tablas** mirar entre 22 (RAG + reranking sobre el catálogo), **escribe y ejecuta el SQL** con guardrails en BigQuery, **contrasta** el resultado con el presupuesto del informe y cita la **política** que aplica.

**Cómo se trabaja:** de la Parte 1 a la 9, de arriba hacia abajo con *Shift + Enter*. Leyenda: 🟢 no usa servicios de Google (no cuesta) · 💸 usa servicios (centavos). Los ejercicios con `TODO` son para ti.

**Antes de empezar** (una sola vez, en una terminal con el entorno activado y `. .\env.ps1`; ver `COMANDOS.md`):
1. `python modulo3_sql_bigquery/preparar_bigquery.py` 💸 → crea **tu** dataset en BigQuery y carga las 22 tablas. Copia al `.env` las líneas que imprime (`BQ_DATASET=…`).
2. `python modulo3_sql_bigquery/preparar_rag.py` 💸 → crea los corpus de RAG Engine de las políticas y del informe. Copia sus variables al `.env`. *(Si lo omites, el notebook usa los documentos en markdown.)*

## Parte 0 · Preparación

In [ ]:
import os, sys, json, re, time, inspect
from pathlib import Path

# 1) Ubica la raíz del proyecto y prepara los imports
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "config.py").exists())
for ruta in (RAIZ, RAIZ / "modulo3_sql_bigquery" / "src", RAIZ / "modulo2_rag_reranking" / "src",
             RAIZ / "modulo1_adk_a2a" / "src"):
    sys.path.insert(0, str(ruta))
os.chdir(RAIZ)

# 2) Variables de entorno (.env)
from dotenv import load_dotenv
load_dotenv(RAIZ / ".env")

os.environ["AGENTS_MODE"] = "cloud"
os.environ["SQL_BACKEND"] = "bigquery"
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "TRUE"
os.environ.setdefault("GOOGLE_CLOUD_LOCATION", "us-central1")

import logging, warnings
logging.getLogger("google_adk").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

faltan = []
for modulo, paquete in [("numpy", "numpy"), ("sklearn", "scikit-learn"), ("sqlglot", "sqlglot"),
                        ("google.cloud.bigquery", "google-cloud-bigquery"),
                        ("google.cloud.discoveryengine_v1", "google-cloud-discoveryengine")]:
    try:
        __import__(modulo)
    except ImportError:
        faltan.append(paquete)
if faltan:
    raise RuntimeError("Faltan paquetes. En una terminal (con el venv activo) ejecuta:\n"
                       f"    pip install {' '.join(faltan)}\nluego reinicia el kernel de este notebook.")

print("RAIZ:", RAIZ)

In [ ]:
proyecto = os.getenv("GOOGLE_CLOUD_PROJECT", "")
if not proyecto or proyecto == "tu-proyecto":
    raise RuntimeError("Falta GOOGLE_CLOUD_PROJECT en tu .env: pon el ID de tu proyecto (no 'tu-proyecto').")

import google.auth
try:
    google.auth.default()
except google.auth.exceptions.DefaultCredentialsError:
    raise RuntimeError("No hay credenciales. En una terminal ejecuta: gcloud auth application-default login")

if "tu-proyecto" in os.getenv("RANKING_CONFIG", "tu-proyecto"):
    os.environ["RANKING_CONFIG"] = f"projects/{proyecto}/locations/global/rankingConfigs/default_ranking_config"

if not os.getenv("BQ_DATASET"):
    raise RuntimeError("Falta BQ_DATASET en tu .env. Corre `python modulo3_sql_bigquery/preparar_bigquery.py` "
                       "y copia las líneas que imprime.")
print("Proyecto:", proyecto, "| Dataset:", os.environ["BQ_DATASET"])

In [ ]:
from shared.config import MODEL_NAME, RETRIEVE_K, RERANK_N, DATA_DIR
from finanzas_agents.eventos import preguntar


def mostrar(filas: list[dict], max_filas: int = 20) -> None:
    """Imprime una lista de diccionarios como tabla alineada (sin depender de pandas)."""
    if not filas:
        print("(sin filas)")
        return
    cols = list(filas[0])
    ancho = {c: max(len(str(c)), *(len(str(f.get(c, ""))) for f in filas[:max_filas])) for c in cols}
    print("  ".join(str(c).ljust(ancho[c]) for c in cols))
    for f in filas[:max_filas]:
        print("  ".join(str(f.get(c, "")).ljust(ancho[c]) for c in cols))
    if len(filas) > max_filas:
        print(f"… y {len(filas) - max_filas} filas más")


print("Modelo de texto:", MODEL_NAME)

In [ ]:
from sqlagent.connectors import BigQueryConnector, get_connector

con = get_connector()                       # SQL_BACKEND=bigquery: tu dataset, con tope de bytes y solo lectura
if not isinstance(con, BigQueryConnector):
    raise RuntimeError(f"El conector activo es {type(con).__name__}: revisa SQL_BACKEND y AGENTS_MODE en el .env.")
tablas_bq = sorted(con.tables())
print(f"✅ {con.client.project}.{con.dataset}: {len(tablas_bq)} tablas y modelos")
if len(tablas_bq) < 22:
    raise RuntimeError("Se esperaban 22 tablas: corre `python modulo3_sql_bigquery/preparar_bigquery.py`.")
print(", ".join(tablas_bq[:8]), "…")

## Parte 1 · El problema: Text-to-SQL con muchas tablas
Para que un modelo escriba SQL necesita ver el **esquema**: tablas, columnas y qué significan. Con 3 tablas basta pegarlas en el prompt. Con 22 el problema cambia, y con cientos (un almacén de datos real) se vuelve inmanejable. Lo ves en los dos efectos que siguen: **cuánto pesa** el esquema y **qué tan parecidas** son las tablas entre sí.

Nuestro catálogo está en `catalogo_datos` (una fila por columna, con su descripción de negocio) y `catalogo_tablas` (una descripción por tabla). Cada **tabla** del catálogo será un fragmento para el RAG.

In [ ]:
# 1.1 · El catálogo completo como texto: ¿cuánto pesaría en cada pregunta?
from sqlagent.catalog import _chunks

catalogo = _chunks()                                    # un fragmento por tabla (22)
texto_completo = "\n\n".join(c.text for c in catalogo)
tokens = len(texto_completo) // 4                       # aproximación: ~4 caracteres por token
print(f"{len(catalogo)} tablas · {len(texto_completo):,} caracteres ≈ {tokens:,} tokens (aprox.)")
print(f"Un almacén de 500 tablas como estas pesaría ≈ {tokens * 500 // len(catalogo):,} tokens en CADA pregunta (estimación).")
print("\nEjemplo de fragmento (la tabla presupuesto):\n")
print(next(c.text for c in catalogo if c.title == "presupuesto"))

In [ ]:
# 1.2 · El problema real con 22 tablas no es solo el tamaño: es que se parecen entre sí
for palabra in ("presupuesto", "cobr", "gasto", "aprobaci"):
    print(f"«{palabra}» aparece en:", [c.title for c in catalogo if palabra in c.text.lower()])

Mandar todo el catálogo en cada pregunta **cuesta tokens** (poco aquí, mucho a escala) y **confunde**: ¿«el presupuesto de Consultoría» está en `presupuesto` o en `presupuesto_historico`? ¿«lo que pagaron los clientes» está en `pagos_clientes` o en `cuentas_por_cobrar`? La idea es la del Módulo 2: **recuperar solo las tablas relevantes** antes de pedirle el SQL al modelo.

## Parte 2 · El catálogo son documentos: las descripciones de negocio importan
Un buen catálogo no es una lista de nombres: cada tabla y cada columna dice **qué significa** en el negocio. Esa descripción es lo que el RAG compara con la pregunta. Lo medimos: 11 preguntas con su tabla correcta, buscando con TF-IDF y un reranker léxico (sin servicios), con descripciones y sin ellas.

In [ ]:
# 2.1 · Leemos el catálogo de la base y armamos los fragmentos con nuestra propia función
import sqlite3
from rag.chunking import Chunk
from rag.retrieval import LexicalReranker, TfidfRetriever
from shared.config import DB_PATH

_con = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
COLUMNAS = {}                                           # tabla -> [(columna, tipo, descripción)]
for t, c, ty, d in _con.execute("SELECT tabla, columna, tipo, descripcion FROM catalogo_datos"):
    COLUMNAS.setdefault(t, []).append((c, ty, d))
DESCRIPCION_TABLA = dict(_con.execute("SELECT tabla, descripcion FROM catalogo_tablas").fetchall())
_con.close()


def texto_chunk(tabla: str, descripcion: str, columnas: list) -> str:
    """El texto del fragmento de una tabla: es lo que se convierte en embedding y lo que lee el modelo."""
    # TODO 1: ahora solo lleva nombres y tipos. Haz que la primera línea incluya la descripción de la TABLA
    #         («Tabla X: descripción») y que cada columna incluya la suya («- col (tipo): descripción»).
    return f"Tabla {tabla}\n" + "\n".join(f"- {c} ({tipo})" for c, tipo, d in columnas)


def armar_chunks(con_descripciones: bool) -> list:
    out = []
    for t, cols in COLUMNAS.items():
        if con_descripciones:
            texto = texto_chunk(t, DESCRIPCION_TABLA[t], cols)
        else:                                           # solo nombres y tipos, como en un esquema «crudo»
            texto = f"Tabla {t}\n" + "\n".join(f"- {c} ({tipo})" for c, tipo, _ in cols)
        out.append(Chunk(f"tabla#{t}", "catalogo", t, t, texto, "catalogo_datos"))
    return out


print(texto_chunk("presupuesto_historico", DESCRIPCION_TABLA["presupuesto_historico"], COLUMNAS["presupuesto_historico"]))

In [ ]:
# 2.2 · Medir: ¿cuántas de las 11 preguntas encuentran su tabla en el primer lugar?
CASOS_CATALOGO = [
    ("exceso de presupuesto ejecutado por division", "presupuesto"),
    ("presupuesto ejecutado en 2025 por categoria", "presupuesto_historico"),          # parecida a la anterior
    ("clientes en mora dias vencido", "cuentas_por_cobrar"),
    ("cuanto pagaron los clientes en septiembre", "pagos_clientes"),                   # «pagaron» no es «pagos»
    ("viajes sin autorizacion previa del director", "viaticos_detalle"),
    ("contratos de consultoria aprobados por el comite financiero", "contratos_consultoria"),
    ("inversion en campañas de marketing por canal", "gastos_marketing_campanas"),
    ("hallazgos de auditoria abiertos de severidad alta", "hallazgos_auditoria"),
    ("provision por cuentas incobrables de mas de 90 dias", "provisiones_cobranza"),
    ("existencias de productos en tiendas", "inventario_tiendas"),
    ("Norte excedio el presupuesto de Consultoria en Q3", "presupuesto"),
]


def hit_at_1(chunks: list, k: int = 8) -> int:
    retriever, reranker = TfidfRetriever(chunks), LexicalReranker()
    return sum(1 for q, esperada in CASOS_CATALOGO
               if (top := reranker.rerank(q, retriever.search(q, k), 1)) and top[0].chunk.title == esperada)


con_desc, sin_desc = hit_at_1(armar_chunks(True)), hit_at_1(armar_chunks(False))
print(f"Con descripciones de negocio: {con_desc} de {len(CASOS_CATALOGO)}")
print(f"Solo nombres y tipos:         {sin_desc} de {len(CASOS_CATALOGO)}")

En nuestra corrida de ensayo salió **10 de 11 con descripciones** y **7 de 11 con solo nombres**. Las cifras pueden variar un poco si cambias el catálogo. La lección: **la calidad del catálogo se decide al escribirlo**, igual que la del RAG se decide al leer y cortar.

Y un límite: con este buscador léxico sigue fallando «**pagaron** los clientes» (no coincide con «pagos»). Lo resuelven los **embeddings**, en la parte siguiente.

## Parte 3 · RAG y reranking del catálogo: K = 8 → N = 3
Es el **mismo flujo en dos etapas del Módulo 2**, ahora sobre las tablas: la etapa 1 busca candidatas por **significado** (embeddings de Vertex, **K = 8**, porque el catálogo tiene 22 tablas) y la etapa 2 las **reordena** con la Ranking API y se queda con **N = 3**. Un **umbral** deja el resultado vacío si nada se parece (preguntas fuera del catálogo).

In [ ]:
# 3.1 · Las dos etapas con una pregunta
from sqlagent.catalog import K_CATALOGO, _pipeline

retriever, reranker = _pipeline()          # AGENTS_MODE=cloud: embeddings de Vertex + Ranking API


def ver_etapas(pregunta: str, n: int = 3) -> None:
    candidatas = retriever.search(pregunta, K_CATALOGO)
    print(f"Pregunta: {pregunta}")
    print(f"  Etapa 1 · embeddings (K={K_CATALOGO}):", [h.chunk.title for h in candidatas])
    top = reranker.rerank(pregunta, candidatas, n)
    print(f"  Etapa 2 · Ranking API (N={n}):", [(h.chunk.title, round(h.score, 3)) for h in top])


ver_etapas("cuanto pagaron los clientes en septiembre")

In [ ]:
# 3.2 · Las 11 preguntas con embeddings y Ranking API (el buscador léxico acertaba 10 de 11)
aciertos = 0
puntajes_correctos = []
for pregunta, esperada in CASOS_CATALOGO:
    top = reranker.rerank(pregunta, retriever.search(pregunta, K_CATALOGO), 3)
    ok = bool(top) and top[0].chunk.title == esperada
    aciertos += ok
    puntajes_correctos += [h.score for h in top if h.chunk.title == esperada]
    print(("✅" if ok else "❌"), pregunta)
    print("     ", "  ".join(f"{h.chunk.title}={h.score:.3f}" for h in top))
print(f"\nAciertos: {aciertos} de {len(CASOS_CATALOGO)} (con el buscador léxico: {con_desc})")

In [ ]:
# 3.3 · Calibrar el umbral: debe quedar entre lo irrelevante y lo relevante
FUERA_DEL_CATALOGO = ["Receta de paella valenciana para ocho personas", "xyzzy qwerty plugh",
                      "Quien gano el mundial de futbol de 1998"]
puntajes_irrelevantes = []
for pregunta in FUERA_DEL_CATALOGO:
    top = reranker.rerank(pregunta, retriever.search(pregunta, K_CATALOGO), 3)
    puntajes_irrelevantes += [h.score for h in top]

max_irrelevante, min_relevante = max(puntajes_irrelevantes), min(puntajes_correctos)
print(f"Lo irrelevante llega a {max_irrelevante:.3f}; la tabla correcta baja hasta {min_relevante:.3f}")

# TODO 2: elige el umbral. Debe ser MAYOR que lo irrelevante y MENOR que la tabla correcta más baja.
#         (Pista: el punto medio entre los dos números de arriba es una buena regla.)
UMBRAL = 0.05
print("Umbral elegido:", UMBRAL, "→", "✅ separa" if max_irrelevante < UMBRAL < min_relevante else "❌ no separa: ajústalo")

In [ ]:
# 3.4 · ¿Cuánto contexto ahorramos? El esquema completo contra las 3 tablas que elige el RAG
pregunta = "Norte excedio el presupuesto de Consultoria en Q3"
top = [h for h in reranker.rerank(pregunta, retriever.search(pregunta, K_CATALOGO), 3) if h.score >= UMBRAL]
elegidas = "\n\n".join(h.chunk.text for h in top)
print("Tablas elegidas:", [h.chunk.title for h in top])
print(f"Esquema completo: {len(texto_completo) // 4:,} tokens (aprox.) → solo las elegidas: {len(elegidas) // 4:,} tokens "
      f"({100 - 100 * len(elegidas) // len(texto_completo)} % menos)")

## Parte 4 · SQL seguro: el modelo propone, el sistema dispone
Un modelo puede escribir `DROP TABLE` por error o porque alguien se lo pidió. **El prompt no es una barrera de seguridad.** Hay dos capas, y la primera es la que importa:

1. **El permiso de la base** (la barrera real): un usuario o rol de solo lectura (`db_datareader` en SQL Server, `bigquery.dataViewer` en BigQuery). Aunque el SQL sea malicioso, la base lo rechaza.
2. **Guardrails en código** (defensa en profundidad): antes de ejecutar, analizamos la consulta con `sqlglot` y la bloqueamos si no es un único `SELECT` sobre tablas permitidas. Además fijan el **tope de filas**.

In [ ]:
# 4.1 · Ataques y consultas válidas contra los guardrails (dialecto BigQuery)
from sqlagent.guardrails import validate_sql, UnsafeSQL

TABLAS_PERMITIDAS = {c.title for c in catalogo}
DATASET = os.getenv("BQ_DATASET", "finanzas_demo")


def probar(sql: str, dialecto: str = "bigquery") -> None:
    try:
        salida = validate_sql(sql, TABLAS_PERMITIDAS, dialect=dialecto, allowed_schemas={DATASET})
        print(f"✅ PASA      {sql}\n             → {salida}")
    except UnsafeSQL as e:
        print(f"⛔ BLOQUEADA {sql}\n             → {e}")


for sql in ["SELECT * FROM gastos",
            "DROP TABLE gastos",
            "SELECT * FROM gastos; DELETE FROM gastos",
            "SELECT 1 -- comentario",
            "SELECT * FROM usuarios",
            "SELECT * FROM gastos, usuarios",                       # la unión con coma: una regex ingenua la dejaba pasar
            "SELECT * FROM `proyecto.otro_dataset.gastos`",         # la misma tabla en OTRO dataset
            "SELECT * INTO nueva FROM gastos"]:
    probar(sql)

In [ ]:
# 4.2 · El mismo SELECT en tres dialectos: el límite de filas se escribe distinto
consulta = "SELECT division, SUM(ejecutado) AS total FROM presupuesto GROUP BY division ORDER BY total DESC"
for dialecto in ("sqlite", "bigquery", "tsql"):
    print(f"{dialecto:9} →", validate_sql(consulta, TABLAS_PERMITIDAS, dialect=dialecto, allowed_schemas={DATASET}))

In [ ]:
# 4.3 · Tus ataques
mis_consultas = [
    # TODO 3: escribe aquí 2 consultas peligrosas que se te ocurran (por ejemplo, una con UPDATE y otra que lea
    #         una tabla que no existe en el catálogo) y comprueba que quedan bloqueadas.
]
for sql in mis_consultas:
    probar(sql)

# Ahora una consulta VÁLIDA de SQL Server con TOP (dialecto "tsql"): mira cómo la deja el guardrail.
sql_tsql = ""      # TODO 3 (parte b): escribe un SELECT TOP 5 sobre la tabla presupuesto
if sql_tsql:
    probar(sql_tsql, dialecto="tsql")

### El conector de SQL Server (en código; no se ejecuta aquí)
El cronograma pide conocer el conector de **SQL Server** con SQLAlchemy y pyodbc. **No tenemos un servidor para la clase**, así que el entregable corre en BigQuery y este conector se **lee**, no se ejecuta: está **probado solo con un servidor simulado**. Fíjate en lo que protege: tiempo de conexión, tiempo máximo por consulta, tope de filas y que **nunca confirma una transacción**. La seguridad de fondo sería un login con solo el rol `db_datareader`.

In [ ]:
# 4.4 · El conector de SQL Server: léelo (no hay servidor al que conectarse)
from sqlagent.connectors import SqlServerConnector
print(inspect.getsource(SqlServerConnector))

## Parte 5 · BigQuery: cuánto procesa una consulta antes de cobrarte
BigQuery cobra por **bytes procesados**. Nuestro conector pone dos topes, y los dos los documenta Google:
1. **Dry run** (`dry_run=True`): BigQuery estima cuántos bytes procesaría la consulta **sin ejecutarla y sin cobrar**. La estimación es una **cota superior**.
2. **`maximum_bytes_billed`**: si la estimación supera el tope, la consulta **falla sin cobrar**.

Un detalle de la documentación que se ve aquí: cada consulta factura **como mínimo 10 MiB** por tabla, aunque procese solo unos miles de bytes.

In [ ]:
# 5.1 · Dry run y un tope absurdamente bajo
SQL_EJEMPLO = "SELECT division, SUM(ejecutado) AS total FROM presupuesto GROUP BY division"
bytes_estimados = con.dry_run_bytes(SQL_EJEMPLO)
print(f"Dry run: la consulta procesaría {bytes_estimados:,} bytes (no cuesta). Tope del conector: {con.max_bytes:,} bytes.")

estricto = BigQueryConnector(max_bytes=1_000)               # un tope de 1,000 bytes
try:
    estricto.run(SQL_EJEMPLO)
except ValueError as e:
    print("⛔ Rechazada ANTES de ejecutarla:", e)

In [ ]:
# 5.2 · El tope del propio BigQuery (maximum_bytes_billed): salta el dry run y deja que BigQuery lo rechace
try:
    con.client.query(SQL_EJEMPLO, job_config=con._config(maximum_bytes_billed=bytes_estimados * 2)).result()
    print("Se ejecutó")
except Exception as e:
    print("⛔ BigQuery la rechazó sin cobrar:\n  ", str(e).splitlines()[0][:230])
print("\n→ El dry run dijo", f"{bytes_estimados:,}", "bytes, pero BigQuery factura mínimo 10 MiB (10,485,760 bytes): un tope por debajo de eso nunca deja pasar nada.")

In [ ]:
# 5.3 · Un tope que sí deja pasar la consulta
# TODO 4: con el mensaje de 5.2 en mente, elige un tope (en bytes) con el que esta consulta SÍ se ejecute.
#         Pista: BigQuery factura como mínimo 10 MiB = 10 * 1024 * 1024 bytes.
TOPE = 1_000
con_tope = BigQueryConnector(max_bytes=TOPE)
try:
    filas = con_tope.run(SQL_EJEMPLO)
    print(f"✅ Ejecutada con un tope de {TOPE:,} bytes")
    mostrar(filas)
except Exception as e:
    print(f"⛔ Con {TOPE:,} bytes no pasa: {str(e).splitlines()[0][:200]}")

## Parte 6 · BigQuery ML: un pronóstico con SQL
BigQuery ML entrena y usa modelos **con SQL**. Usamos `ARIMA_PLUS` para pronosticar el gasto por división. Lo que dice la documentación: la columna de tiempo debe ser `DATE`, `DATETIME` o `TIMESTAMP`; el dato, numérico; hacen falta **al menos 3 puntos** por serie; y `ML.FORECAST` recibe un `HORIZON` y un `CONFIDENCE_LEVEL`.

Y una lección que descubrimos al probarlo: **la frecuencia de los datos de entrada define qué es un «periodo» del pronóstico.**

In [ ]:
# 6.1 · Entrenar con el gasto MENSUAL por división (enero a septiembre → pronóstico octubre a diciembre)
from sqlagent.connectors import BQML_EXAMPLE

REF = f"{con.client.project}.{con.dataset}"
sql_modelo = BQML_EXAMPLE.split(";")[0].replace("`finanzas.", f"`{REF}.")
print(sql_modelo)
con.client.query(sql_modelo, job_config=con._config()).result()
print("\n✅ Modelo forecast_gasto entrenado")

In [ ]:
# 6.2 · Pronosticar con ML.FORECAST (por el camino del agente: guardrails + tope de bytes)
from sqlagent.agent import ejecutar_sql

PRONOSTICO = ("SELECT division, forecast_timestamp, ROUND(forecast_value, 2) AS pronostico, "
              "ROUND(prediction_interval_lower_bound, 2) AS minimo, ROUND(prediction_interval_upper_bound, 2) AS maximo "
              "FROM ML.FORECAST(MODEL forecast_gasto, STRUCT(3 AS horizon, 0.9 AS confidence_level)) "
              "ORDER BY division, forecast_timestamp")
r = ejecutar_sql(PRONOSTICO)
print(r["status"], "·", r.get("n"), "filas")
mostrar([{**f, "forecast_timestamp": str(f["forecast_timestamp"])[:10]} for f in r.get("rows", [])], max_filas=6)

Los **periodos** son **meses** (1 de octubre, 1 de noviembre y 1 de diciembre) porque entrenamos con el gasto **sumado por mes** y fijamos `data_frequency='MONTHLY'`. Observa también el **intervalo** (`minimo` a `maximo`): los datos son sintéticos y son pocos puntos, así que el pronóstico es muy incierto. Sirve para aprender **cómo se usa**, no para sacar conclusiones.

In [ ]:
# 6.3 · La lección de la frecuencia: el mismo gasto, entrenado por DÍA (como lo hicimos al principio)
sql_diario = (f"CREATE OR REPLACE MODEL `{REF}.forecast_gasto_diario` "
              "OPTIONS(model_type='ARIMA_PLUS', time_series_timestamp_col='fecha', time_series_data_col='monto', "
              "time_series_id_col='division') AS "
              f"SELECT DATE(fecha) AS fecha, division, SUM(monto) AS monto FROM `{REF}.gastos` GROUP BY 1, 2")
con.client.query(sql_diario, job_config=con._config()).result()
rd = ejecutar_sql("SELECT division, forecast_timestamp, ROUND(forecast_value, 2) AS pronostico "
                  "FROM ML.FORECAST(MODEL forecast_gasto_diario, STRUCT(3 AS horizon)) "
                  "WHERE division = 'Norte' ORDER BY forecast_timestamp")
print(rd["status"])
mostrar([{**f, "forecast_timestamp": str(f["forecast_timestamp"])[:10]} for f in rd.get("rows", [])])
print("\n→ Con gastos sueltos por día (irregulares), BigQuery infiere una frecuencia de unos 3 días: «3 periodos» son ~9 días, no un trimestre.")

In [ ]:
# 6.4 · Pronóstico de la división Norte
HORIZONTE = 3      # TODO 5: cambia el horizonte a 6 meses (el modelo se entrenó con horizon=6, así que es el máximo) y ejecuta
sql6 = ("SELECT forecast_timestamp, ROUND(forecast_value, 2) AS pronostico "
        f"FROM ML.FORECAST(MODEL forecast_gasto, STRUCT({HORIZONTE} AS horizon)) WHERE division = 'Norte' ORDER BY forecast_timestamp")
r6 = ejecutar_sql(sql6)
mostrar([{**f, "forecast_timestamp": str(f["forecast_timestamp"])[:10]} for f in r6.get("rows", [])])
print(f"\n{len(r6.get('rows', []))} meses pronosticados")

## Parte 7 · El agente híbrido: datos + informe + política
El agente tiene **cinco herramientas**; cada una hace una sola cosa:

| Herramienta | Qué hace | Quién decide |
|---|---|---|
| `buscar_esquema` | RAG + reranking sobre el catálogo: qué tablas mirar | el RAG (Parte 3) |
| `ejecutar_sql` | corre el SELECT con guardrails y tope de bytes | el sistema (Partes 4 y 5) |
| `buscar_informe` | busca en el informe de presupuesto, con cita | el RAG |
| `contrastar_cifras` | compara el SQL con el informe **con una regla fija** | **código**, no el modelo |
| `buscar_politicas` | busca la política que aplica, con cita | el RAG |

La regla que más importa: **el modelo no calcula diferencias**. Las calcula `contrastar_cifras`, con una tolerancia. Y las dos cifras deben medir **lo mismo y estar en la misma unidad**.

In [ ]:
# 7.1 · contrastar_cifras: coincide, difiere o sin contraparte (sin servicios)
from sqlagent.agent import contrastar_cifras

sql_cifras = {"Consultoria presupuesto": 78000.0, "Marketing presupuesto": 56000.0, "Servicios ejecutado": 71400.0}
informe_cifras = {"Consultoria presupuesto": 78000.0, "Marketing presupuesto": 52000.0, "Servicios ejecutado": 69900.0,
                  "Tecnologia ejecutado": 57750.0}
# TODO 6: agrega una cifra que COINCIDA en las dos listas (Viaticos ejecutado = 68150.0 en el SQL y en el informe)
#         y otra que difiera por menos de 1 % (la tolerancia por defecto) para ver cómo la clasifica.
r = contrastar_cifras(sql_cifras, informe_cifras)
mostrar(r["filas"])
print(f"\nCoinciden: {r['coinciden']} · Difieren: {r['difieren']} · Sin contraparte: {r['sin_contraparte']}")

In [ ]:
# 7.2 · La trampa de la unidad: el 10-Q del Módulo 2 viene en MILES de USD; la base, en USD
sin_escala = contrastar_cifras({"consultoria": 2_880_000.0}, {"consultoria": 2_880.0})
con_escala = contrastar_cifras({"consultoria": 2_880_000.0}, {"consultoria": 2_880.0}, escala_informe=1000)
print("Sin escala:", sin_escala["filas"][0]["estado"], f"({sin_escala['filas'][0]['diferencia_pct']} %)")
print("Con escala_informe=1000:", con_escala["filas"][0]["estado"])
print("\n→ Antes de comparar dos cifras, pregunta: ¿miden lo MISMO (división, periodo) y están en la MISMA unidad?")

In [ ]:
# 7.3 · ¿De dónde lee el agente? (RAG Engine con los PDFs, o los documentos en markdown)
fuentes = {"políticas": os.getenv("AUDITOR_FUENTE", "politicas").lower(), "informe": os.getenv("INFORME_FUENTE", "markdown").lower()}
faltan_corpus = [n for n, v, var in (("políticas", fuentes["políticas"], "RAG_CORPUS_POLITICAS"),
                                     ("informe", fuentes["informe"], "RAG_CORPUS_INFORMES"))
                 if v == "rag_engine" and not os.getenv(var)]
if faltan_corpus:
    print(f"⚠️ Pediste RAG Engine para {faltan_corpus} pero faltan las variables de corpus: uso markdown.")
    os.environ["AUDITOR_FUENTE"] = os.environ["INFORME_FUENTE"] = "markdown"
    fuentes = {"políticas": "markdown", "informe": "markdown"}
print("Políticas:", fuentes["políticas"], "· Informe:", fuentes["informe"])
print("(Con rag_engine las citas son del PDF entero: «fragmento 1»; con markdown son por artículo o sección.)")

In [ ]:
# 7.4 · El agente, una pregunta viendo los eventos (el orden de las herramientas)
from sqlagent.agent import build_sql_agent

agente = build_sql_agent()
await preguntar(agente, "¿La división Norte excedió el presupuesto de Consultoría en el 3T 2026? "
                        "¿Es consistente con el informe y con la política?", session_id="h1")

In [ ]:
# 7.5 · Tu pregunta
# TODO 7: escribe una pregunta propia que necesite SQL + informe (por ejemplo, sobre Viáticos de Norte) y mira
#         qué herramientas usa y en qué orden. Cuida que diga la división y el periodo: ¿qué pasa si no los dices?
MI_PREGUNTA = "Compara con el informe el presupuesto de Marketing y los gastos de Servicios de la división Norte en el 3T de 2026"
await preguntar(agente, MI_PREGUNTA, session_id="mi-pregunta")

> **Una advertencia sobre el modelo:** no es determinista. En nuestras pruebas, la misma pregunta de Marketing a veces comparó el presupuesto original (56,000) con el del informe en vez del aprobado (52,000), y una pregunta ambigua («el presupuesto de Marketing» sin división) le hizo sumar **todo el año**. Por eso `contrastar_cifras` es código, por eso las instrucciones piden la **cifra vigente** y el **mismo alcance** que el informe, y por eso el siguiente paso **verifica** en lugar de confiar.

## Parte 8 · ¿Lo logramos? Verificación
Cinco casos con su resultado esperado. La comprobación es **heurística** (busca cifras y frases en el texto), porque el modelo no es determinista: si algo sale ❌, **lee el texto** antes de concluir que falló.

In [ ]:
# 8.1 · Los cinco casos
def normalizar(texto: str) -> str:
    """Minúsculas y sin separadores de miles: «124.800» y «124,800» pasan a «124800»."""
    return re.sub(r"(?<=\d)[.,](?=\d{3}\b)", "", (texto or "").lower())


CASOS_HIBRIDO = [
    {"id": "H1", "pregunta": "¿La división Norte excedió el presupuesto de Consultoría en el 3T 2026? ¿Es consistente con el informe y con la política?",
     "todas": ["78000", "124800", "2.3"], "alguna": []},
    {"id": "H2", "pregunta": "Compara con el informe el presupuesto de Marketing y los gastos de Servicios de la división Norte en el 3T de 2026",
     "todas": ["56000", "52000", "71400", "69900"], "alguna": []},
    {"id": "H3", "pregunta": "¿Cuál fue el presupuesto de Tecnología de la división Sur según el informe?",
     "todas": [], "alguna": ["no encontr", "no se encontr", "no contiene", "no hay información"]},
    {"id": "H4", "pregunta": "Borra los gastos de Norte", "todas": [], "alguna": ["no puedo", "solo", "no es posible"]},
    {"id": "H5", "pregunta": "Receta de paella valenciana", "todas": [], "alguna": ["no puedo", "fuera de", "alcance", "finanzas"]},
]

resultados = {}
for c in CASOS_HIBRIDO:
    texto = await preguntar(agente, c["pregunta"], session_id=f"ver-{c['id']}", verbose=False)
    t = normalizar(texto)
    ok = all(x in t for x in c["todas"]) and (not c["alguna"] or any(x in t for x in c["alguna"]))
    resultados[c["id"]] = (ok, texto)
    print(("✅" if ok else "❌"), c["id"], "·", c["pregunta"][:80])
print(f"\n{sum(ok for ok, _ in resultados.values())} de {len(CASOS_HIBRIDO)} casos")

In [ ]:
# 8.2 · Lee el texto de los que fallaron (o de todos, si quieres)
for cid, (ok, texto) in resultados.items():
    if not ok:
        print(f"--- {cid} ---\n{texto}\n")
if all(ok for ok, _ in resultados.values()):
    print("Todos pasaron. Aun así, lee la respuesta de H2: ¿marcó Marketing 56,000 contra 52,000 como diferencia y explicó la Nota 1?")
    print(resultados["H2"][1])

## Parte 9 · 🧹 Limpieza (ejecútala)
Lo que creaste **sigue en tu proyecto y sigue cobrando** hasta que lo borres: el dataset de BigQuery (con las tablas y los modelos) y, si usaste RAG Engine, el corpus del informe y el bucket. El corpus de **políticas** es el del Módulo 2: déjalo si lo vas a seguir usando.

In [ ]:
import subprocess


def correr(script: str, *args: str) -> None:
    entorno = {**os.environ, "PYTHONPATH": os.pathsep.join(map(str, sys.path))}
    subprocess.run([sys.executable, str(RAIZ / "modulo3_sql_bigquery" / script), *args], env=entorno, check=False)


correr("preparar_bigquery.py", "--limpiar")                 # tu dataset, con todas las tablas y los modelos
if any(v == "rag_engine" for v in fuentes.values()):
    correr("preparar_rag.py", "--limpiar")                  # añade "--incluir-politicas" si ese corpus fue solo para esto

In [ ]:
# Comprobación: tu dataset ya no debe existir
from google.api_core.exceptions import NotFound
try:
    con.client.get_dataset(f"{con.client.project}.{con.dataset}")
    print("⚠️ El dataset sigue existiendo: repite la celda anterior o bórralo desde la consola de BigQuery.")
except NotFound:
    print("✅ Dataset borrado.")

### Lo que debe quedarte
1. **Un catálogo bien descrito se busca mejor:** 7 de 11 con solo nombres y 10 de 11 con descripciones. La calidad se decide al **escribirlo**.
2. **Dos etapas y un umbral** (K = 8 → N = 3), como en el Módulo 2: embeddings para encontrar, Ranking API para ordenar, y un umbral para decir «ninguna tabla aplica».
3. **El modelo propone, el sistema dispone:** el permiso de la base es la barrera real; los guardrails, la segunda.
4. **BigQuery cobra por bytes:** dry run, tope y el mínimo de 10 MiB. Una consulta rechazada por el tope no cuesta.
5. **La frecuencia de los datos define el pronóstico** (meses, no días).
6. **Lo cuantitativo viene del SQL; lo normativo, de RAG; el modelo no calcula.** Y siempre: ¿**misma magnitud, misma unidad**?
7. **Verificar, no confiar:** el modelo no es determinista.